In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import sys
import json

outputs_dir = Path("outputs")
csv_files = list(outputs_dir.glob("*_memory_stats.csv"))

print(f"Found {len(csv_files)} CSV files:")
for f in csv_files:
    print(f"  - {f.name}")

if len(csv_files) == 0:
    print("No CSV files found in outputs/ directory!")
    sys.exit(0)

print("\n" + "="*60)

# Children of insert_allocation — these columns hold JSON with bytes/bytes_top
CHILD_COLS = ['new_allocation', 'grow_amortized', 'grow_one', 'finish_grow', 'add_name', 'tree_new']

cmap = plt.cm.Set3
color_map = {name: cmap(i) for i, name in enumerate(CHILD_COLS + ['Other MiriMachine', 'Non-Miri'])}

def parse_bytes_top(cell) -> float:
    """Extract bytes_top from a JSON cell, return 0.0 on failure."""
    try:
        return float(json.loads(cell).get('bytes_top', 0))
    except Exception:
        return 0.0

def parse_bytes(cell) -> float:
    """Extract bytes (not top) from a JSON cell, return 0.0 on failure."""
    try:
        return float(json.loads(cell).get('bytes', 0))
    except Exception:
        return 0.0

def to_mb(b: float) -> float:
    return b / (1024 * 1024)

for csv_path in csv_files:
    project = csv_path.stem.replace("_memory_stats", "")

    print(f"\n{'='*60}")
    print(f"PROCESSING: {project.upper()}")
    print(f"{'='*60}\n")

    df = pd.read_csv(csv_path)

    print(f"Total test cases: {len(df)}")
    print(f"Configs: {df['config'].unique().tolist()}")

    configs = sorted(df['config'].unique())
    plot_configs = [c for c in configs if c != 'default']

    print(f"\n{'='*60}")
    print("AVERAGES BY CONFIG (bytes_top of insert_alloc children vs MiriMachine bytes)")
    print('='*60)

    summary = []
    for cfg in plot_configs:
        cfg_df = df[df['config'] == cfg]

        row = {'config': cfg, 'count': len(cfg_df)}

        # bytes_top for each child (mean across test cases)
        for col in CHILD_COLS:
            if col in cfg_df.columns:
                row[f'{col}_top_mb'] = cfg_df[col].apply(parse_bytes_top).mean() / (1024*1024)
            else:
                row[f'{col}_top_mb'] = 0.0

        # MiriMachine total bytes (not top) as the denominator/baseline
        if 'miri_machine' in cfg_df.columns:
            row['miri_bytes_mb'] = cfg_df['miri_machine'].apply(parse_bytes).mean() / (1024*1024)
        else:
            row['miri_bytes_mb'] = 0.0

        # Total bytes across all allocations
        row['total_bytes_mb'] = cfg_df['total_bytes'].mean() / (1024*1024) if 'total_bytes' in cfg_df.columns else 0.0
        row['non_miri_mb'] = max(0.0, row['total_bytes_mb'] - row['miri_bytes_mb'])

        children_total = sum(row[f'{col}_top_mb'] for col in CHILD_COLS)
        row['other_miri_mb'] = max(0.0, row['miri_bytes_mb'] - children_total)

        summary.append(row)

        total = row['miri_bytes_mb']
        print(f"\n{cfg.upper()} (MiriMachine bytes = {total:.2f} MB):")
        for col in CHILD_COLS:
            v = row[f'{col}_top_mb']
            print(f"  {col:20s}: {v:8.2f} MB ({v/total*100:5.1f}% of MiriMachine)")
        print(f"  {'Other MiriMachine':20s}: {row['other_miri_mb']:8.2f} MB ({row['other_miri_mb']/total*100:5.1f}%)")
        print(f"  {'Non-Miri':20s}: {row['non_miri_mb']:8.2f} MB ({row['non_miri_mb']/row['total_bytes_mb']*100:5.1f}% of total)")

    print(f"\n{'='*60}")
    print("GENERATING PIE CHARTS")
    print('='*60)

    if not summary:
        print("No configs to plot!")
    else:
        fig, axes = plt.subplots(1, len(summary), figsize=(6 * len(summary), 6))
        if len(summary) == 1:
            axes = [axes]

        fig.suptitle(f'{project.upper()} — insert_allocation children (bytes_top) vs MiriMachine bytes', fontsize=13)

        for idx, row in enumerate(summary):
            ax = axes[idx]
            cfg = row['config']

            labels, sizes, colors = [], [], []
            for col in CHILD_COLS:
                v = row[f'{col}_top_mb']
                if v > 0 and not np.isnan(v):
                    labels.append(col)
                    sizes.append(v)
                    colors.append(color_map[col])

            other = row['other_miri_mb']
            if other > 0 and not np.isnan(other):
                labels.append('Other MiriMachine')
                sizes.append(other)
                colors.append(color_map['Other MiriMachine'])

            non_miri = row['non_miri_mb']
            if non_miri > 0 and not np.isnan(non_miri):
                labels.append('Non-Miri')
                sizes.append(non_miri)
                colors.append(color_map['Non-Miri'])

            if not sizes:
                print(f"Skipping {cfg} — no data")
                continue

            ax.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
            ax.set_title(f'{cfg.upper()}\nMiriMachine total: {row["miri_bytes_mb"]:.1f} MB')

        plt.tight_layout()
        pie_path = outputs_dir / f'{project}_insert_alloc_children_pies.png'
        plt.savefig(pie_path, dpi=150, bbox_inches='tight')
        plt.close()
        print(f"Saved: {pie_path}")

Found 3 CSV files:
  - socket2-outputs_memory_stats.csv
  - libc-outputs_memory_stats.csv
  - getrandom-outputs_memory_stats.csv


PROCESSING: SOCKET2-OUTPUTS

Total test cases: 30
Configs: ['miri', 'miri-tree', 'default']

AVERAGES BY CONFIG (bytes_top of insert_alloc children vs MiriMachine bytes)

MIRI (MiriMachine bytes = 338.82 MB):
  new_allocation      :    72.06 MB ( 21.3% of MiriMachine)
  grow_amortized      :   134.78 MB ( 39.8% of MiriMachine)
  grow_one            :    24.13 MB (  7.1% of MiriMachine)
  finish_grow         :     6.03 MB (  1.8% of MiriMachine)
  add_name            :     0.00 MB (  0.0% of MiriMachine)
  tree_new            :     0.00 MB (  0.0% of MiriMachine)
  Other MiriMachine   :   101.81 MB ( 30.0%)
  Non-Miri            :   130.65 MB ( 27.8% of total)

MIRI-TREE (MiriMachine bytes = 431.45 MB):
  new_allocation      :     5.58 MB (  1.3% of MiriMachine)
  grow_amortized      :   134.53 MB ( 31.2% of MiriMachine)
  grow_one            :    24.29 MB (

In [ ]:
# import pandas as pd
# import matplotlib.pyplot as plt
# import numpy as np
# from pathlib import Path
# import sys

# outputs_dir = Path("outputs")
# csv_files = list(outputs_dir.glob("*_memory_stats.csv"))

# print(f"Found {len(csv_files)} CSV files:")
# for f in csv_files:
#     print(f"  - {f.name}")

# if len(csv_files) == 0:
#     print("No CSV files found in outputs/ directory!")
#     sys.exit(0)

# print("\n" + "="*60)

# cmap = plt.cm.Set3
# component_names = ['Insert Alloc', 'Retag', 'Eval Args', 'Prov GC', 'Tree Transition', 'Other Miri', 'Non-Miri']
# color_map = {name: cmap(i) for i, name in enumerate(component_names)}

# for csv_path in csv_files:
#     project = csv_path.stem.replace("_memory_stats", "")
    
#     print(f"\n{'='*60}")
#     print(f"PROCESSING: {project.upper()}")
#     print(f"{'='*60}\n")
    
#     df = pd.read_csv(csv_path)
    
#     # Convert bytes to MB
#     byte_cols = [c for c in df.columns if 'bytes' in c]
#     for col in byte_cols:
#         df[f'{col}_mb'] = df[col] / (1024 * 1024)
    
#     print(f"Total test cases: {len(df)}")
#     print(f"Configs: {df['config'].unique().tolist()}")
    
#     print(f"\n{'='*60}")
#     print("AVERAGES BY CONFIG")
#     print('='*60)
    
#     configs = sorted(df['config'].unique())
#     summary = []
#     plot_configs = [c for c in configs if c != 'default']

#     for cfg in plot_configs:
#         cfg_df = df[df['config'] == cfg]
        
#         data = {
#             'config': cfg,
#             'count': len(cfg_df),
#             'total_mb': cfg_df['total_bytes_mb'].mean(),
#             'insert_alloc_mb': cfg_df['insert_allocation_bytes_mb'].mean(),
#             'retag_mb': cfg_df['b_retag_bytes_mb'].mean() if 'b_retag_bytes_mb' in cfg_df.columns else 0,
#             'eval_args_mb': cfg_df['eval_callee_and_args_bytes_mb'].mean(),
#             'prov_gc_mb': cfg_df['provenance_gc_bytes_mb'].mean(),
#             'tree_transition_mb': cfg_df['perform_transition_bytes_mb'].mean() if 'perform_transition_bytes_mb' in cfg_df.columns else 0,
#             'miri_mb': cfg_df['mirimachine_bytes_mb'].mean(),
#         }
        
#         tracked = (data['insert_alloc_mb'] + data['retag_mb'] + 
#                    data['eval_args_mb'] + data['prov_gc_mb'] + data['tree_transition_mb'])
#         data['other_mb'] = data['miri_mb'] - tracked
#         data['non_miri_mb'] = data['total_mb'] - data['miri_mb']
        
#         summary.append(data)
    
#     summary_df = pd.DataFrame(summary)
    
#     print("\n### Memory Usage (MB and %) ###")
#     for cfg in plot_configs:
#         row = summary_df[summary_df['config'] == cfg].iloc[0]
#         total = row['total_mb']
        
#         print(f"\n{cfg.upper()}:")
#         print(f"  Insert Allocation:  {row['insert_alloc_mb']:8.2f} MB ({row['insert_alloc_mb']/total*100:5.1f}%)")
#         print(f"  Retag:              {row['retag_mb']:8.2f} MB ({row['retag_mb']/total*100:5.1f}%)")
#         print(f"  Eval Args:          {row['eval_args_mb']:8.2f} MB ({row['eval_args_mb']/total*100:5.1f}%)")
#         print(f"  Provenance GC:      {row['prov_gc_mb']:8.2f} MB ({row['prov_gc_mb']/total*100:5.1f}%)")
#         if row['tree_transition_mb'] > 0:
#             print(f"  Tree Transition:    {row['tree_transition_mb']:8.2f} MB ({row['tree_transition_mb']/total*100:5.1f}%)")
#         print(f"  Other MiriMachine:  {row['other_mb']:8.2f} MB ({row['other_mb']/total*100:5.1f}%)")
#         print(f"  Total MiriMachine:  {row['miri_mb']:8.2f} MB ({row['miri_mb']/total*100:5.1f}%)")
#         print(f"  Non-Miri:           {row['non_miri_mb']:8.2f} MB ({row['non_miri_mb']/total*100:5.1f}%)")
#         print(f"  TOTAL:              {total:8.2f} MB")
    
#     print(f"\n{'='*60}")
#     print("GENERATING PIE CHARTS")
#     print('='*60)
    
#     if len(plot_configs) == 0:
#         print("No configs with MiriMachine data to plot!")
#     else:
#         fig, axes = plt.subplots(1, len(plot_configs), figsize=(6*len(plot_configs), 6))
#         if len(plot_configs) == 1:
#             axes = [axes]
            
#         fig.suptitle(f'{project.upper()}', fontsize=14)
        
#         for idx, cfg in enumerate(plot_configs):
#             row = summary_df[summary_df['config'] == cfg].iloc[0]
            
#             labels = []
#             sizes = []
#             colors = []
            
#             components = [
#                 ('Insert Alloc', row['insert_alloc_mb']),
#                 ('Retag', row['retag_mb']),
#                 ('Eval Args', row['eval_args_mb']),
#                 ('Prov GC', row['prov_gc_mb']),
#                 ('Tree Transition', row['tree_transition_mb']),
#                 ('Other Miri', row['other_mb']),
#                 ('Non-Miri', row['non_miri_mb'])
#             ]
            
#             for label, size in components:
#                 if size > 0 and not np.isnan(size):
#                     labels.append(label)
#                     sizes.append(size)
#                     colors.append(color_map[label])
            
#             if not sizes or all(s == 0 for s in sizes):
#                 print(f"Skipping {cfg} - no data")
#                 continue
            
#             ax = axes[idx]
#             ax.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
#             ax.set_title(f'{cfg.upper()}\n{row["total_mb"]:.1f} MB')
        
#         plt.tight_layout()
#         pie_path = outputs_dir / f'{project}_pies.png'
#         plt.savefig(pie_path, dpi=150, bbox_inches='tight')
#         plt.close()
#         print(f"Saved: {pie_path}")

Found 3 CSV files:
  - socket2-outputs_memory_stats.csv
  - libc-outputs_memory_stats.csv
  - getrandom-outputs_memory_stats.csv


PROCESSING: SOCKET2-OUTPUTS

Total test cases: 30
Configs: ['miri', 'miri-tree', 'default']

AVERAGES BY CONFIG

### Memory Usage (MB and %) ###

MIRI:
  Insert Allocation:    216.47 MB ( 46.1%)
  Retag:                 25.26 MB (  5.4%)
  Eval Args:             38.40 MB (  8.2%)
  Provenance GC:          0.24 MB (  0.1%)
  Other MiriMachine:     58.45 MB ( 12.5%)
  Total MiriMachine:    338.82 MB ( 72.2%)
  Non-Miri:             130.65 MB ( 27.8%)
  TOTAL:                469.47 MB

MIRI-TREE:
  Insert Allocation:    203.06 MB ( 36.0%)
  Retag:                 41.87 MB (  7.4%)
  Eval Args:             38.88 MB (  6.9%)
  Provenance GC:          1.74 MB (  0.3%)
  Tree Transition:       72.39 MB ( 12.8%)
  Other MiriMachine:     73.50 MB ( 13.0%)
  Total MiriMachine:    431.45 MB ( 76.5%)
  Non-Miri:             132.43 MB ( 23.5%)
  TOTAL:                56

In [ ]:
# # Time analysis by config
# print(f"\n{'='*60}")
# print("TIME ANALYSIS")
# print('='*60)

# for csv_path in csv_files:
#     project = csv_path.stem.replace("_memory_stats", "")
    
#     print(f"\n{'='*60}")
#     print(f"PROJECT: {project.upper()}")
#     print(f"{'='*60}\n")
    
#     df = pd.read_csv(csv_path)
    
#     # Convert timestamp to seconds
#     def timestamp_to_seconds(ts):
#         parts = str(ts).split(':')
#         if len(parts) == 4:  # HH:MM:SS:MS
#             hours, mins, secs, ms = parts
#             return int(hours) * 3600 + int(mins) * 60 + int(secs) + float(ms)
#         return 0
    
#     df['time_seconds'] = df['time'].apply(timestamp_to_seconds)
    
#     # Calculate average time per config
#     time_by_config = df.groupby('config')['time_seconds'].mean().sort_index()
    
#     print("Average execution time by config:")
#     baseline = None
#     for cfg, avg_time in time_by_config.items():
#         if cfg == 'default':
#             baseline = avg_time
#             print(f"  {cfg:15s}: {avg_time:8.3f} seconds (1.0x)")
#         else:
#             multiplier = avg_time / baseline if baseline else 1.0
#             print(f"  {cfg:15s}: {avg_time:8.3f} seconds ({multiplier:.1f}x)")
    
#     # Create bar chart
#     fig, ax = plt.subplots(figsize=(8, 6))
    
#     configs = time_by_config.index.tolist()
#     times = time_by_config.values
    
#     # Use colors from the same Set3 colormap
#     bar_colors = [cmap(8), cmap(6), cmap(9)][:len(configs)]
    
#     bars = ax.bar(configs, times, color=bar_colors)
    
#     # Add value labels - time on top, multiplier in middle of bar
#     baseline = time_by_config.get('default', time_by_config.iloc[0])
#     for bar, time_val in zip(bars, times):
#         height = bar.get_height()
#         multiplier = time_val / baseline
        
#         # Time label on top
#         ax.text(bar.get_x() + bar.get_width()/2., height,
#                 f'{height:.3f}s',
#                 ha='center', va='bottom', fontsize=10)
        
#         # Multiplier label in middle of bar
#         if multiplier == 1.0:
#             label = ''
#         else:
#             label = f'{multiplier:.1f}x'
#         ax.text(bar.get_x() + bar.get_width()/2., height/2,
#                 label,
#                 ha='center', va='center', fontsize=11, fontweight='bold')
    
#     ax.set_ylabel('Time (seconds)', fontsize=12)
#     ax.set_title(f'{project.upper()}', fontsize=14)
    
#     plt.tight_layout()
#     time_bar_path = outputs_dir / f'{project}_time_bars.png'
#     plt.savefig(time_bar_path, dpi=150, bbox_inches='tight')
#     plt.close()
#     print(f"\nSaved: {time_bar_path}")

# print(f"\n{'='*60}")
# print("ALL TIME CHARTS GENERATED")
# print('='*60)


TIME ANALYSIS

PROJECT: SOCKET2-OUTPUTS

Average execution time by config:
  default        :    0.487 seconds (1.0x)
  miri           :   48.925 seconds (100.4x)
  miri-tree      :   56.575 seconds (116.1x)

Saved: outputs/socket2-outputs_time_bars.png

PROJECT: LIBC-OUTPUTS

Average execution time by config:
  default        :    0.283 seconds (1.0x)
  miri           :   52.940 seconds (187.2x)
  miri-tree      :   56.495 seconds (199.7x)

Saved: outputs/libc-outputs_time_bars.png

PROJECT: GETRANDOM-OUTPUTS

Average execution time by config:
  default        :    0.301 seconds (1.0x)
  miri           :   33.700 seconds (112.0x)
  miri-tree      :   33.480 seconds (111.3x)

Saved: outputs/getrandom-outputs_time_bars.png

ALL TIME CHARTS GENERATED
